In [2]:
!pip install pycolmap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 53.0 MB/s eta 0:00:00:00:0100:01


### **PREFACE**

Our main goal is to reconstruct 3D object(in our case, church) from set of images.

In this version, we are going to recreate structure from motion by classic computer vision algorithms such as SIFT and BFMatching. Also we will use brute-force due to relatively small size of set of images. And for the record, in the next versions will be image comparison by similarity of embeddings, and usage of nerual networks for the keypoint detection and comparison. 

In [20]:
import warnings
warnings.filterwarnings('ignore') 

In [4]:
import numpy as np

import cv2 
import os
import itertools

import tqdm
from pathlib import Path

import h5py
import plotly.graph_objects as go

import pycolmap

import sys

# Insert the folder path (exclude the file name itself)
sys.path.insert(1, '/kaggle/input/datasets/konivlarov/tgaafaf')

# Import your module normally
from database import *
from h5py_file import *

In [8]:
def detect_keypoints(
    paths: list[Path],
    feature_dir: Path
) -> None:
    #dtype = torch.float32 # ALIKED has issues with float16
    
    extractor = cv2.SIFT_create()
    
    feature_dir.mkdir(parents=True, exist_ok=True)

    
    with h5py.File(feature_dir / "keypoints.h5", mode="w") as f_keypoints, \
         h5py.File(feature_dir / "descriptors.h5", mode="w") as f_descriptors:
        
        for path in tqdm(paths, desc="Computing keypoints"):
            key = path.name
            
            image = cv2.imread(path)
            
            kp, ds = extractor.detectAndCompute(image, None)
            '''
            kp = [{
                        'pt': p.pt,
                        'size': p.size,
                        'angle': p.angle,
                        'response': p.response,
                        'octave': p.octave,
                        'class_id': p.class_id
                        } for p in kp]
            '''
            kp = np.array([kps.pt for kps in kp])
            f_keypoints[key] = kp
            f_descriptors[key] = ds

    
def keypoint_distances(
    paths: list[Path],
    index_pairs: list[tuple[int, int]],
    feature_dir: Path,
    min_matches: int = 15,
    verbose: bool = False,
    #device: torch.device = torch.device("cpu"),
) -> None:

    matcher = cv2.BFMatcher()
    
    with h5py.File(feature_dir / "keypoints.h5", mode="r") as f_keypoints, \
         h5py.File(feature_dir / "descriptors.h5", mode="r") as f_descriptors, \
         h5py.File(feature_dir / "matches.h5", mode="w") as f_matches:
        
            for idx1, idx2 in tqdm(index_pairs, desc="Computing keypoing distances"):
                key1, key2 = paths[idx1].name, paths[idx2].name

                matches = matcher.knnMatch(f_descriptors[key1][...], 
                                   f_descriptors[key2][...], k=2)
    
                queryIdx, trainIdx = [], []
            
                for m,n in matches:
                    if m.distance < 0.85*n.distance:
                        queryIdx.append(m.queryIdx), trainIdx.append(m.trainIdx)
                    
                indices = np.column_stack((queryIdx, trainIdx))
                # We have matches to consider
                n_matches = len(trainIdx)
                
                if n_matches:
                    if verbose:
                        print(f"{key1}-{key2}: {n_matches} matches")
                    # Store the matches in the group of one image
                    if n_matches >= min_matches:
                        group  = f_matches.require_group(key1)
                        group.create_dataset(key2, data=indices)


def import_into_colmap(
    path: Path,
    feature_dir: Path,
    database_path: str = "colmap.db",
) -> None:
    """Adds keypoints into colmap"""
    db = COLMAPDatabase.connect(database_path)
    db.create_tables()
    single_camera = False
    fname_to_id = add_keypoints(db, feature_dir, path, "", "simple-pinhole", single_camera)
    add_matches(
        db,
        feature_dir,
        fname_to_id,
    )
    db.commit()

In [9]:
path = '/kaggle/input/competitions/image-matching-challenge-2024/test/church/images'
feature_dir = Path("./sample_test_features")
images_list = list(Path(path).glob("*.png"))

detect_keypoints(images_list, feature_dir)
all_pairs = list(itertools.combinations([i for i in range(len(images_list))], 2))
keypoint_distances(images_list, all_pairs, feature_dir)

Computing keypoing distances: 100%|██████████| 820/820 [03:56<00:00,  3.47it/s]


In [10]:
database_path = "colmap.db"
images_dir = images_list[0].parent
import_into_colmap(
    images_dir, 
    feature_dir, 
    database_path
)

# This does RANSAC
pycolmap.match_exhaustive(database_path)

100%|██████████| 41/41 [00:01<00:00, 28.79it/s]
820it [00:00, 3146.86it/s]                         
I20260825 09:20:43.241151 139491377411648 feature_matching.cc:195] === Feature matching & geometric verification ===
I20260825 09:20:43.241992 139490420536896 sift.cc:1565] Creating SIFT CPU feature matcher
I20260825 09:20:43.242029 139490052003392 sift.cc:1565] Creating SIFT CPU feature matcher
I20260825 09:20:43.242256 139490068788800 sift.cc:1565] Creating SIFT CPU feature matcher
I20260825 09:20:43.242388 139490060396096 sift.cc:1565] Creating SIFT CPU feature matcher
I20260825 09:20:43.242814 139491377411648 pairing.cc:180] Generating exhaustive image pairs...
I20260825 09:20:43.242954 139491377411648 pairing.cc:213] Processing block [1/1, 1/1]
I20260825 09:21:37.683114 139491377411648 feature_matching.cc:217] in 54.440s
I20260825 09:21:37.683216 139491377411648 timer.cc:90] Elapsed time: 0.907 [minutes]


In [15]:
mapper_options = pycolmap.IncrementalPipelineOptions()
mapper_options.min_model_size = 8
mapper_options.max_num_models = 10

maps = pycolmap.incremental_mapping(
    database_path=database_path, 
    image_path=images_dir,
    output_path=Path.cwd() / "incremental_pipeline_outputs"
)
maps[0]

I20260825 09:57:09.722849 139492830310400 incremental_pipeline.cc:278] Loading database
I20260825 09:57:09.722961 139492830310400 database_cache.cc:72] Loading rigs...
I20260825 09:57:09.723007 139492830310400 database_cache.cc:82]  0 in 0.000s
I20260825 09:57:09.723028 139492830310400 database_cache.cc:90] Loading cameras...
I20260825 09:57:09.723132 139492830310400 database_cache.cc:108]  41 in 0.000s
I20260825 09:57:09.723146 139492830310400 database_cache.cc:116] Loading frames...
I20260825 09:57:09.723167 139492830310400 database_cache.cc:126]  0 in 0.000s
I20260825 09:57:09.723178 139492830310400 database_cache.cc:134] Loading matches...
I20260825 09:57:09.725106 139492830310400 database_cache.cc:139]  820 in 0.002s
I20260825 09:57:09.725136 139492830310400 database_cache.cc:147] Loading images...
I20260825 09:57:09.730900 139492830310400 database_cache.cc:241]  41 in 0.006s (connected 41, loaded 41)
I20260825 09:57:09.730953 139492830310400 database_cache.cc:255] Loading pose pr

Reconstruction(num_rigs=38, num_cameras=38, num_frames=38, num_reg_frames=38, num_images=38, num_points3D=2714)

In [18]:

def visualize(maps: pycolmap.Reconstruction) -> None:

  xyz = []
  rgb = []
  
  for point3D_id, point3D in maps.points3D.items():
      xyz.append(point3D.xyz)
      # PyCOLMAP stores colors as 0-255 integers; Open3D requires 0.0-1.0 floats
      rgb.append(point3D.color / 255.0)
      
  xyz_np = np.array(xyz)
  rgb_np = np.array(rgb)
  
  fig = go.Figure(data=[go.Scatter3d(
      x=xyz_np[:, 0], y=xyz_np[:, 1], z=xyz_np[:, 2],
      mode='markers',
      marker=dict(
          size=2,
          color=['rgb({},{},{})'.format(int(r*255), int(g*255), int(b*255)) for r, g, b in rgb_np],
          opacity=0.8
      )
  )])
  
  fig.update_layout(margin=dict(l=0, r=0, b=0, t=0))
  fig.show()

visualize(maps[0])